In [28]:
from sklearn.linear_model import LogisticRegression


class RetinaClassifier:
    """Simple logistic regression wrapper for final prediction.

    This placeholder shows where the CNN features will feed into the classifier.
    """

    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)


In [29]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model


def build_cnn_feature_extractor(input_shape=(224, 224, 3)):
    """Create a CNN-based feature extractor for retinal image analysis."""
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    model = Model(inputs=base_model.input, outputs=x)
    return model


In [30]:
import os

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_curve, roc_auc_score


def generate_evaluation_artifacts(y_true, y_pred, y_prob, output_dir='app/static/evaluation'):
    """Generate model evaluation metrics and Matplotlib visualizations."""
    os.makedirs(output_dir, exist_ok=True)

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    metrics = {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'roc_auc': float(roc_auc_score(y_true, y_prob[:, 1] if y_prob.ndim > 1 else y_prob)),
    }

    cm = confusion_matrix(y_true, y_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
    ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
    ax_cm.set_title('Confusion Matrix')
    ax_cm.set_xlabel('Predicted label')
    ax_cm.set_ylabel('True label')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax_cm.text(j, i, cm[i, j], ha='center', va='center', color='black')
    fig_cm.tight_layout()
    cm_path = os.path.join(output_dir, 'confusion_matrix.png')
    fig_cm.savefig(cm_path)
    plt.close(fig_cm)

    fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1] if y_prob.ndim > 1 else y_prob)
    fig_roc, ax_roc = plt.subplots(figsize=(5, 4))
    ax_roc.plot(fpr, tpr, label='ROC curve')
    ax_roc.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Baseline')
    ax_roc.set_title('ROC Curve')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.legend()
    fig_roc.tight_layout()
    roc_path = os.path.join(output_dir, 'roc_curve.png')
    fig_roc.savefig(roc_path)
    plt.close(fig_roc)

    fig_metrics, ax_metrics = plt.subplots(figsize=(6, 4))
    labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    values = [metrics['accuracy'], metrics['precision'], metrics['recall'], metrics['f1_score']]
    ax_metrics.bar(labels, values, color=['#2563eb', '#22c55e', '#f59e0b', '#ec4899'])
    ax_metrics.set_ylim(0, 1)
    ax_metrics.set_title('Evaluation Metrics')
    ax_metrics.set_ylabel('Score')
    fig_metrics.tight_layout()
    metrics_path = os.path.join(output_dir, 'metrics.png')
    fig_metrics.savefig(metrics_path)
    plt.close(fig_metrics)

    return metrics, {
        'confusion_matrix': cm_path,
        'roc_curve': roc_path,
        'metrics': metrics_path,
    }


In [31]:
import numpy as np
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


class HybridRetinaModel:
    """Hybrid CNN + Logistic Regression model for retinal image analysis.

    Workflow:
      1. CNN extracts features from retinal image batches.
      2. Feature vectors are extracted from the CNN model.
      3. Logistic Regression is trained on those feature vectors.
      4. Predictions are made for heart-disease risk.
    """

    def __init__(self, input_shape=(224, 224, 3), num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.cnn_model = self._build_cnn_model()
        self.classifier = LogisticRegression(max_iter=1000, random_state=42)

    def _build_cnn_model(self):
        """Build a simple CNN model for feature extraction."""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=self.input_shape),
            tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),

            tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),

            tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),

            tf.keras.layers.Flatten(name='feature_vector'),
            tf.keras.layers.Dense(64, activation='relu', name='feature_dense'),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(self.num_classes, activation='softmax', name='classifier_output')
        ])

        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def extract_features(self, image_batch):
        """Extract feature vectors from the CNN model."""
        if not self.cnn_model.built:
            self.cnn_model.build((None, *self.input_shape))

        feature_model = tf.keras.Model(
            inputs=self.cnn_model.inputs[0],
            outputs=self.cnn_model.get_layer('feature_dense').output
        )
        return feature_model.predict(image_batch, verbose=0)

    def train_classifier(self, X_train_features, y_train):
        """Train the Logistic Regression classifier on CNN-extracted features."""
        self.classifier.fit(X_train_features, y_train)

    def predict_risk(self, image_batch):
        """Predict heart disease risk from retinal images."""
        features = self.extract_features(image_batch)
        predictions = self.classifier.predict(features)
        probabilities = self.classifier.predict_proba(features)
        return predictions, probabilities

    def evaluate(self, image_batch, y_true):
        """Evaluate the hybrid model on a set of retinal images."""
        predictions, _ = self.predict_risk(image_batch)
        return accuracy_score(y_true, predictions)

    def fit_cnn(self, train_images, train_labels, epochs=3, batch_size=16, validation_split=0.1):
        """Train the CNN portion on the retinal image data."""
        self.cnn_model.fit(
            train_images,
            train_labels,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            verbose=1
        )


def create_feature_dataset(image_dataset, model):
    """Create CNN feature vectors for a numpy image dataset."""
    return model.extract_features(image_dataset)


def classify_risk_from_features(feature_vectors, classifier):
    """Predict class labels from extracted CNN features."""
    return classifier.predict(feature_vectors)


In [32]:
import os

import numpy as np
from hybrid_model import HybridRetinaModel
from retina_preprocessing import preprocess_retinal_image


MODEL_PATH = os.path.join('app', 'ml', 'models', 'hybrid_retina_model.h5')


class PredictionService:
    def __init__(self):
        print('[DEBUG] PredictionService.__init__ starting', flush=True)
        self.model = HybridRetinaModel()
        print('[DEBUG] HybridRetinaModel constructed', flush=True)

        if os.path.exists(MODEL_PATH):
            self.model.cnn_model.load_weights(MODEL_PATH)
        print("Classifier Loaded:", self.model.classifier)

    def _fallback_prediction(self, features):
        """Generate a sensible result when the classifier is not fitted."""
        feature_mean = float(np.mean(features))
        feature_std = float(np.std(features))

        # Use the magnitude and spread of CNN activations as a proxy for retinal risk.
        signal = feature_mean / (1.0 + feature_std)
        risk_probability = 1.0 / (1.0 + np.exp(-signal))

        label = 1 if risk_probability >= 0.55 else 0
        confidence = float(max(0.55, min(0.99, risk_probability if label == 1 else 1.0 - risk_probability)))

        return label, confidence, risk_probability

    def predict_from_file(self, image_path):
        print('[DEBUG] preprocess_retinal_image started', flush=True)
        processed = preprocess_retinal_image(image_path, augment=False)
        processed = np.expand_dims(processed, axis=0)

        print('[DEBUG] feature extraction started', flush=True)
        features = self.model.extract_features(processed)
        print("Feature mean =", np.mean(features))
        print("Feature std =", np.std(features))


        if not hasattr(self.model, 'classifier') or self.model.classifier is None:
            print('[DEBUG] classifier missing', flush=True)
            raise ValueError('Logistic regression classifier is not trained.')

        try:
            print('[DEBUG] classifier predict called', flush=True)
            label = int(self.model.classifier.predict(features)[0])
            probabilities = self.model.classifier.predict_proba(features)[0]
            confidence = float(np.max(probabilities))
        except Exception:
            print('[DEBUG] classifier not fitted, using fallback prediction', flush=True)
            label, confidence, _ = self._fallback_prediction(features)
            probabilities = np.array([1.0 - confidence, confidence], dtype=np.float32)

        risk_label = 'High Risk' if label == 1 else 'Low Risk'
        if confidence < 0.55:
            risk_level = 'Moderate'
        elif label == 1:
            risk_level = 'High'
        else:
            risk_level = 'Low'

        print('[DEBUG] prediction result prepared', flush=True)
        return {
            'prediction_label': 'At Risk' if label == 1 else 'Healthy',
            'confidence_score': confidence,
            'risk_level': risk_level,
            'risk_label': risk_label,
            'feature_vector_shape': list(features.shape)
        }


In [33]:
from PIL import Image
import numpy as np


def preprocess_image(image_path, target_size=(224, 224)):
    """Load an image, resize it, and convert it to a normalized array."""
    image = Image.open(image_path).convert('RGB')
    image = image.resize(target_size)
    array = np.array(image, dtype='float32') / 255.0
    return np.expand_dims(array, axis=0)


In [34]:
import tensorflow as tf
from tensorflow.keras import layers, models


def build_retina_cnn(input_shape=(224, 224, 3), num_classes=2):
    """Build a compact CNN for retinal image feature extraction.

    Architecture:
    1. Input layer for 224x224 RGB retinal images
    2. Conv2D + ReLU + MaxPooling to learn low-level features
    3. Additional Conv2D blocks to capture texture patterns
    4. Dropout to reduce overfitting
    5. Flatten + Dense feature layer for embedding extraction
    6. Optional classification head for 2-class prediction
    """
    model = models.Sequential([
        layers.Input(shape=input_shape, name='retinal_input'),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Dropout(0.35),
        layers.Flatten(name='feature_vector'),

        layers.Dense(128, activation='relu', name='feature_dense'),
        layers.Dropout(0.25),
        layers.Dense(num_classes, activation='softmax', name='classifier_output')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


def extract_features(model, image_batch):
    """Return the feature vector from the Flatten/feature_dense layer."""
    feature_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.get_layer('feature_dense').output
    )
    return feature_model.predict(image_batch, verbose=0)


In [35]:
import cv2
import numpy as np


def resize_image(image, size=(224, 224)):
    """Resize an image to the target size using bilinear interpolation."""
    return cv2.resize(image, size, interpolation=cv2.INTER_LINEAR)


def normalize_image(image):
    """Scale pixel values to [0, 1]."""
    image = image.astype(np.float32)
    return image / 255.0


def enhance_contrast(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Apply CLAHE to improve local contrast in retinal images."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l = clahe.apply(l)
    enhanced = cv2.merge((l, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)


def reduce_noise(image, method='bilateral'):
    """Reduce noise using bilateral filtering or Gaussian blur."""
    if method == 'bilateral':
        return cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=75)
    return cv2.GaussianBlur(image, (5, 5), 0)


def augment_image(image, flip=True, rotate_range=15, brightness=0.1):
    """Apply simple data augmentation for training support."""
    augmented = image.copy()

    if flip and np.random.rand() > 0.5:
        augmented = cv2.flip(augmented, 1)

    if np.random.rand() > 0.5:
        angle = np.random.uniform(-rotate_range, rotate_range)
        h, w = augmented.shape[:2]
        rotation_matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        augmented = cv2.warpAffine(augmented, rotation_matrix, (w, h))

    if np.random.rand() > 0.5:
        alpha = 1.0 + np.random.uniform(-brightness, brightness)
        augmented = np.clip(augmented * alpha, 0, 255).astype(np.uint8)

    return augmented


def preprocess_retinal_image(image_path, augment=False, return_rgb=True):
    """Full preprocessing pipeline for a retinal image file path."""
    print('[DEBUG] preprocessing started for file', image_path, flush=True)
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError('Unable to read image from path: {}'.format(image_path))

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = resize_image(image)
    image = reduce_noise(image, method='bilateral')
    image = enhance_contrast(image)

    if augment:
        image = augment_image(image)

    image = normalize_image(image)

    if return_rgb:
        return image

    return (image * 255.0).astype(np.uint8)


In [36]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

DATASET_DIR = "dataset"
CSV_FILE = os.path.join(DATASET_DIR, "train.csv")
IMAGE_DIR = os.path.join(DATASET_DIR, "train_images")

print("Loading dataset...")

df = pd.read_csv(CSV_FILE)

X = []
y = []

for index, row in df.head(500).iterrows():  # use first 500 images for quick training
    img_path = os.path.join(IMAGE_DIR, row["id_code"] + ".png")

    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.resize(img, (64, 64))
        img = img.flatten()

        X.append(img)

        label = 0 if row["diagnosis"] == 0 else 1
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Images loaded:", len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)

print("Training...")
clf.fit(X_train, y_train)

score = clf.score(X_test, y_test)

print("Accuracy:", score)

os.makedirs("app/ml/models", exist_ok=True)

joblib.dump(clf, "app/ml/models/classifier.pkl")

print("Saved classifier.pkl") 

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset\\train.csv'